# Deep RL with Stable-Baselines3 - what RL Lab runs under the hood

In the RL Lab interface you trained DQN and PPO agents with one click. Under the hood, the backend is a thin wrapper around [stable-baselines3](https://stable-baselines3.readthedocs.io/) (SB3) - and the core of what it does fits in about fifteen lines, which you will run yourself in this notebook.

This notebook is deliberately minimal. It is a guided tour, not a course - each section links to the official documentation where the real depth lives:

- [Gymnasium documentation](https://gymnasium.farama.org) - the environment API (you used it in the Q-Learning notebook)
- [SB3 documentation](https://stable-baselines3.readthedocs.io/) and the [SB3 RL tutorial](https://stable-baselines3.readthedocs.io/en/master/guide/rl_tips.html)
- [rl-baselines3-zoo](https://github.com/DLR-RM/rl-baselines3-zoo) - community-tuned hyperparameters per environment (RL Lab's defaults come from here)
- [Hugging Face Deep RL Course](https://huggingface.co/learn/deep-rl-course/unit0/introduction) - free, hands-on, highly recommended as a next step

## 0. One-time install of the deep-RL dependencies

**The next cell installs the OPTIONAL bigger dependencies** for this notebook (PyTorch + stable-baselines3, roughly 500 MB, CPU-only - no GPU needed). The basic environment from the Q-Learning notebook stays lightweight; only this notebook needs the extras.

Just run the cell - no terminal required. It is safe to re-run (already-installed packages are skipped). If you prefer the terminal instead: `uv sync --extra sb3` in the `examples/` folder.

In [ ]:
# OPTIONAL bigger dependencies for this notebook (~500 MB, CPU-only).
# Safe to re-run. Terminal equivalent:  uv sync --extra sb3
%pip install --quiet torch --index-url https://download.pytorch.org/whl/cpu
%pip install --quiet "stable-baselines3>=2.6.0" tqdm rich  # tqdm+rich power the training progress bar
print("Install finished. If this was the FIRST run: restart the kernel once (Kernel > Restart), then continue below.")

## 1. The environment

Same [Gymnasium](https://gymnasium.farama.org) API as in the Q-Learning notebook - but [CartPole](https://gymnasium.farama.org/environments/classic_control/cart_pole/) has a **continuous** state (4 floating-point numbers), so a Q-table is impossible. That is the entire reason neural networks enter the picture.

In [ ]:
import gymnasium as gym

env = gym.make('CartPole-v1', render_mode='rgb_array')

obs, info = env.reset(seed=42)
print("Observation space:", env.observation_space)   # 4 continuous numbers -> no table possible
print("Action space:     ", env.action_space)        # 2 discrete actions: push left / push right
print("One observation:  ", obs)

## 2. The agent - three lines

This is, quite literally, what RL Lab's backend builds when you click Start Training. The hyperparameters below are the community-tuned CartPole values from [rl-baselines3-zoo](https://github.com/DLR-RM/rl-baselines3-zoo/blob/master/hyperparams/dqn.yml) - the exact defaults you saw in RL Lab's sliders.

In [ ]:
from stable_baselines3 import DQN

model = DQN(
    'MlpPolicy', env,
    learning_rate=0.0023,          # rl-zoo tuned values for CartPole -
    gamma=0.99,                    # the same numbers RL Lab uses
    batch_size=64,
    exploration_fraction=0.16,
    exploration_final_eps=0.04,
    buffer_size=100_000,
    learning_starts=1000,
    target_update_interval=10,
    train_freq=256,
    gradient_steps=128,
    policy_kwargs={'net_arch': [256, 256]},
    seed=42,
    verbose=0,
)

## 3. The standard RL workflow: evaluate → train → evaluate

Exactly what RL Lab's **Evaluate Policy** button does: run N episodes greedily (no exploration) and report the average return. Evaluating *before* training gives the baseline.

In [ ]:
from stable_baselines3.common.evaluation import evaluate_policy

mean_before, std_before = evaluate_policy(model, env, n_eval_episodes=20, deterministic=True)
print(f"Untrained policy: {mean_before:.1f} +/- {std_before:.1f}  (random flailing, pole falls fast)")

In [ ]:
# ~2-4 minutes on a CPU. This is RL Lab's Start Training button.
model.learn(total_timesteps=50_000, progress_bar=True)

In [ ]:
mean_after, std_after = evaluate_policy(model, env, n_eval_episodes=20, deterministic=True)
print(f"Trained policy:   {mean_after:.1f} +/- {std_after:.1f}  (CartPole counts as solved at >= 475; max 500)")

## 4. Watch one episode

RL Lab's **Play Policy** is a rollout with `deterministic=True`, rendering each step. Here we just show the final frame - for the animated version, use RL Lab itself.

In [ ]:
import matplotlib.pyplot as plt

obs, info = env.reset(seed=7)
terminated = truncated = False
steps = 0
while not (terminated or truncated):
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = env.step(action)
    steps += 1

plt.imshow(env.render())
plt.axis('off')
plt.title(f"Final frame after {steps} steps (500 = survived the full episode)")
plt.show()

## Exercise 1 - the hard environment

Rebuild the model above with `gym.make('MountainCar-v0', render_mode='rgb_array')` but **keep the CartPole hyperparameters**, train 50k steps and evaluate. You should see the agent stuck at -200 (it never reaches the flag).

Then look up the zoo's tuned [MountainCar values](https://github.com/DLR-RM/rl-baselines3-zoo/blob/master/hyperparams/dqn.yml) (they are also in RL Lab's `backend/algorithms/dqn.py`), rebuild with those, and train `total_timesteps=100_000` (~5 minutes). 

*Question to think about: why does a single lucky success change everything for DQN, but not for PPO? Hint: what does the replay buffer remember?*

## Exercise 2 - same API, different algorithm family

Swap `DQN` for `PPO` (`from stable_baselines3 import PPO; model = PPO('MlpPolicy', env, seed=42)`) - SB3's defaults are fine on CartPole - and repeat the evaluate → train (50k) → evaluate workflow.

Then try PPO on MountainCar and compare with Exercise 1. In RL Lab, the MountainCar environment page explains why this one is expected to fail.

Solutions to both exercises: `solutions/solution_sb3_quickstart.ipynb`.

## Where to go next

- [SB3 examples & tutorial](https://stable-baselines3.readthedocs.io/en/master/guide/examples.html) - saving/loading models, callbacks, custom environments
- [Gymnasium basic usage](https://gymnasium.farama.org/introduction/basic_usage/) - the environment API in depth
- [rl-baselines3-zoo](https://github.com/DLR-RM/rl-baselines3-zoo) - train any algorithm on any environment with tuned settings, one command
- [Hugging Face Deep RL Course](https://huggingface.co/learn/deep-rl-course/unit0/introduction) - the best free structured path onward
- Sutton & Barto, [Reinforcement Learning: An Introduction](http://incompleteideas.net/book/the-book-2nd.html) - the theory, free PDF